# Environment and data audit

Standalone, clean-kernel execution with checkpointed prerequisites. Synthetic mode is explicit. See docs/RESEARCH_PROTOCOL.md for scientific definitions. Final test access is guarded.

In [1]:
import os

if os.environ.get("BRFSS_SYNTHETIC", "0") != "1":
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ.setdefault(
        "BRFSS_REPO",
        "/content/drive/MyDrive/BRFSS2015_Statistical_Methods_AZ/BRFSS2015-Statistical-Methods",
    )

In [2]:
import os
import sys
from pathlib import Path

REPO = Path(os.environ.get("BRFSS_REPO", Path.cwd()))
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").exists():
    raise FileNotFoundError("Set BRFSS_REPO to the extracted repository folder.")
os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
SYNTHETIC = os.environ.get("BRFSS_SYNTHETIC", "0") == "1"
OUT = Path(
    os.environ.get(
        "BRFSS_OUT",
        "runs/notebook_synthetic"
        if SYNTHETIC
        else "/content/drive/MyDrive/BRFSS2015_Statistical_Methods_AZ/runs/primary",
    )
)
DATA_PATH = os.environ.get("BRFSS_DATA")
CODEBOOK_PATH = os.environ.get("BRFSS_CODEBOOK")
PROFILE = os.environ.get("BRFSS_PROFILE", "quick")
MODE = os.environ.get("BRFSS_MODE", "binary")

In [3]:
if not SYNTHETIC:
    from google.colab import drive

    drive.mount("/content/drive")
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
from scripts.verify_environment import verify

versions = verify()

No broken requirements found.
{
  "python": "3.12.14",
  "numpy": "2.2.6",
  "pandas": "2.2.3",
  "scipy": "1.15.3",
  "scikit-learn": "1.6.1",
  "statsmodels": "0.14.4",
  "imbalanced-learn": "0.13.0",
  "matplotlib": "3.10.1",
  "seaborn": "0.13.2",
  "PyYAML": "6.0.2",
  "pyreadstat": "1.2.8",
  "nbformat": "5.10.4",
  "nbclient": "0.10.2",
  "ipykernel": "6.29.5",
  "pytest": "8.3.5",
  "ruff": "0.11.13",
  "requests": "2.32.3",
  "pypdf": "5.5.0"
}


In [4]:
from src.pipeline import run

run(
    OUT,
    data=DATA_PATH,
    codebook=CODEBOOK_PATH,
    synthetic=SYNTHETIC,
    n=960,
    profile=PROFILE,
    mode=MODE,
    through="audit",
)

2026-11-06 08:04:59,492 INFO RESUME audit


PosixPath('/tmp/tmpomboo_1a')

In [5]:
import json

artifact = json.loads((OUT / "audit.json").read_text())
print(json.dumps(artifact, indent=2, default=str)[:5000])

{
  "source_columns": [
    "SEQNO",
    "_STATE",
    "_PSU",
    "_STSTR",
    "_LLCPWT",
    "_AGEG5YR",
    "SEX",
    "_BMI5",
    "EDUCA",
    "INCOME2",
    "_RACE",
    "BPHIGH4",
    "TOLDHI2",
    "_SMOKER3",
    "EXERANY2",
    "_FRUTSUM",
    "_VEGESUM",
    "_DRNKWEK",
    "GENHLTH",
    "PHYSHLTH",
    "MENTHLTH",
    "HLTHPLN1",
    "MEDCOST",
    "CHECKUP1",
    "DIABETE3",
    "PREDIAB1"
  ],
  "source_sha256": "fd43384ff0c09934bf4b6bad99d8bada01c851a3deadf9d8847e24f0867b3f4c",
  "records": 960,
  "columns_loaded": 26,
  "strata": 120,
  "psus_nested": 480,
  "design_df": 360,
  "duplicate_ids": 0,
  "duplicate_predictor_rows": 0,
  "outcome_counts": {
    "3": 729,
    "1": 144,
    "4": 87
  },
  "synthetic": true,
  "codebook": "SYNTHETIC_FIXTURE_NO_CDC_CODING_VERIFICATION"
}
